In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from groq import Groq

load_dotenv(find_dotenv())
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"

print("Key loaded:", os.getenv("GROQ_API_KEY") is not None)

Key loaded: True


In [2]:
def get_llm_answer(question):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer the question in 4-6 factual sentences."},
            {"role": "user", "content": question}
        ],
        temperature=0.7,
        max_tokens=1000
    )
    return response.choices[0].message.content

question = "Tell me about the life and career of Rosalind Franklin."
answer = get_llm_answer(question)
print(answer)

Rosalind Franklin (1920‑1958) was a British chemist and X‑ray crystallographer whose work was pivotal in uncovering the structure of DNA. She earned her Ph.D. at Cambridge University, studying the physical chemistry of coal, and later joined King's College London, where she produced the famous “Photo 51” X‑ray diffraction image of DNA. After moving to Birkbeck College, she applied her expertise to the structure of viruses, notably the tobacco mosaic virus, advancing the field of molecular biology. Franklin’s meticulous research and analytical techniques laid essential groundwork for the double‑helix model, although she did not receive full recognition during her lifetime. She died of ovarian cancer at age 37, but her contributions are now celebrated as critical to modern genetics.


In [3]:
import json, re

def extract_claims(text):
    prompt = f"""Break the text below into independent, atomic factual claims.

Rules:
- One fact per claim.
- Replace pronouns (he, she, it, they) with the full name.
- Each claim must make sense on its own.
- Return ONLY a JSON list of strings, nothing else.

Text:
{text}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=1500
    )
    output = response.choices[0].message.content

    match = re.search(r"\[.*\]", output, re.DOTALL)
    return json.loads(match.group(0)) if match else []

claims = extract_claims(answer)
for i, c in enumerate(claims, 1):
    print(i, c)

1 Rosalind Franklin was born in 1920.
2 Rosalind Franklin died in 1958.
3 Rosalind Franklin was a British chemist.
4 Rosalind Franklin was a British X‑ray crystallographer.
5 Rosalind Franklin's work was pivotal in uncovering the structure of DNA.
6 Rosalind Franklin earned her Ph.D. at Cambridge University.
7 Rosalind Franklin studied the physical chemistry of coal for her Ph.D.
8 Rosalind Franklin later joined King's College London.
9 Rosalind Franklin produced the "Photo 51" X‑ray diffraction image of DNA at King's College London.
10 Rosalind Franklin moved to Birkbeck College.
11 Rosalind Franklin applied her expertise to the structure of viruses at Birkbeck College.
12 Rosalind Franklin studied the tobacco mosaic virus.
13 Rosalind Franklin advanced the field of molecular biology.
14 Rosalind Franklin's meticulous research and analytical techniques laid essential groundwork for the double‑helix model.
15 Rosalind Franklin did not receive full recognition during her lifetime.
16 Ro

In [4]:
import wikipediaapi

wiki = wikipediaapi.Wikipedia(user_agent="hallucination-detector/1.0 (student project)", language="en")

page = wiki.page("Rosalind Franklin")
print("Page exists:", page.exists())
print("Text length:", len(page.text))
print(page.text[:500])

Page exists: True
Text length: 74440
Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer. Her work was central to the understanding of the molecular structures of DNA (deoxyribonucleic acid), RNA (ribonucleic acid), viruses, coal, and graphite. Although her works on coal and viruses were appreciated in her lifetime, Franklin's contributions to the discovery of the structure of DNA were largely unrecognised during her life, for which Franklin has been variously referred to as the 


In [5]:
import re
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# split the Wikipedia page into sentences
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", page.text) if len(s.strip()) > 30]
sentence_embeddings = embedder.encode(sentences, convert_to_tensor=True)
print("Sentences in page:", len(sentences))

def get_evidence(claim, top_k=3):
    claim_embedding = embedder.encode(claim, convert_to_tensor=True)
    scores = util.cos_sim(claim_embedding, sentence_embeddings)[0]
    top = scores.topk(top_k)
    return [(sentences[i], round(float(s), 3)) for s, i in zip(top.values, top.indices)]

for idx in [0, 8, 22]:
    print("\nCLAIM:", claims[idx])
    for sent, score in get_evidence(claims[idx]):
        print(f"  [{score}] {sent[:200]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentences in page: 497

CLAIM: Rosalind Franklin was born in 1920.
  [0.862] "Rosalind Franklin (1920–1958)".
  [0.828] "Rosalind Elsie Franklin 1920–1958".
  [0.75] Rosalind Elsie Franklin (25 July 1920 – 16 April 1958) was an English chemist and X-ray crystallographer.

CLAIM: Rosalind Franklin produced the "Photo 51" X‑ray diffraction image of DNA at King's College London.
  [0.765] The other contribution included a photograph of an X-ray diffraction pattern of B-DNA (called Photo 51), taken by Franklin's student Gosling, that was briefly shown to Watson by Wilkins in January 195
  [0.759] Franklin is best known for her work on the X-ray diffraction images of DNA while at King's College London, particularly Photo 51, taken by her student Raymond Gosling, which led to the discovery of th
  [0.745] Franklin was fully committed to experimental data and was sternly against theoretical or model buildings, as she said, "We are not going to speculate, we are going to wait, we are going to 

IndexError: list index out of range